# Chapter 13: Pooling Cross Sections Across Time — Simple Panel Data Methods

Topics: pooled cross sections, difference-in-differences, organizing panel data, first-differenced estimator.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 13.1 Pooled Cross Sections

## Step 2 – Pooled Cross-Sections

Pooled cross-sections stack independent samples from different time periods.

$$y_{it} = \beta_0 + \delta_1 d_{t2} + \cdots + \delta_{T} d_{tT} + \mathbf{x}_{it}'\boldsymbol{\beta} + u_{it}$$

where $d_{ts}$ is a dummy for period $s$. The year dummies control for aggregate time trends that affect everyone.

Note: pooled cross-sections have **different individuals** in each period (unlike panel data which tracks the same individuals).

In [ ]:
df = woo.dataWoo('cps78_85')
print(df.head())
print(f"\nYears in data: {df['year'].unique()}")

# Pool both years, control for year effect
m_pool = smf.ols('np.log(wage) ~ educ + exper + female + C(year)', data=df).fit()
print(m_pool.summary())

## 13.2 Difference-in-Differences

## Step 3 – Difference-in-Differences (DiD)

DiD estimates the **causal effect** of a policy/treatment by comparing:
- The change over time for the **treatment group** (before → after)
- The change over time for the **control group** (before → after)

**DiD regression model:**
$$y_{it} = \beta_0 + \beta_1 \,dB_i + \beta_2 \,d2_t + \beta_3 \,(dB_i \cdot d2_t) + u_{it}$$

| | Before ($d2=0$) | After ($d2=1$) | Difference |
|---|---|---|---|
| Control ($dB=0$) | $\beta_0$ | $\beta_0+\beta_2$ | $\beta_2$ |
| Treatment ($dB=1$) | $\beta_0+\beta_1$ | $\beta_0+\beta_1+\beta_2+\beta_3$ | $\beta_2+\beta_3$ |
| **DiD** | | | $\hat{\beta}_3$ |

**Identifying assumption:** parallel trends — in the absence of treatment, both groups would have followed the same time path.

In [ ]:
df = woo.dataWoo('kielmc')

# DiD: effect of incinerator on house prices
# treated = nearinc, post = y81
m_did = smf.ols('rprice ~ nearinc + y81 + nearinc:y81', data=df).fit()
print(m_did.summary())

did_estimate = m_did.params['nearinc:y81']
print(f"\nDiD Estimate: ${did_estimate:.2f}")
print(f"Houses near incinerator fell by ${abs(did_estimate):.2f} relative to control, after 1981")

## DiD Visualisation

## Step 4 – First-Differenced Estimator

For two-period panel data, first-differencing eliminates the individual fixed effect $a_i$:

$$y_{it} = \mathbf{x}_{it}'\boldsymbol{\beta} + a_i + u_{it}$$

Taking $\Delta = $ (period 2) $-$ (period 1):
$$\Delta y_i = \Delta\mathbf{x}_i'\boldsymbol{\beta} + \Delta u_i$$

The individual effect $a_i$ cancels — this is the key advantage of panel data for controlling for unobserved time-constant heterogeneity.

In [ ]:
means = df.groupby(['nearinc', 'y81'])['rprice'].mean().reset_index()
means['group'] = means['nearinc'].map({0: 'Control (far)', 1: 'Treated (near)'})
means['period'] = means['y81'].map({0: 'Pre (1978)', 1: 'Post (1981)'})

fig, ax = plt.subplots(figsize=(8, 5))
for group, gdf in means.groupby('group'):
    ax.plot(gdf['period'].values, gdf['rprice'].values, marker='o', lw=2, label=group, markersize=8)

ax.set_ylabel('Mean House Price ($)')
ax.set_title('Difference-in-Differences: House Prices')
ax.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_did_plot.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_13_Pooled_Cross_Sections_DiD_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 13.4 First-Differenced Estimator

In [ ]:
df = woo.dataWoo('crime2')
df['d_crmrte'] = df['crmrte'] - df.groupby('county')['crmrte'].shift(1)
df['d_unem']   = df['unem']   - df.groupby('county')['unem'].shift(1)

m_fd = smf.ols('d_crmrte ~ d_unem', data=df.dropna()).fit()
print("First-Differenced Estimator:")
print(m_fd.summary())